# Aether Paper Artifact on Kaggle

Import this notebook into Kaggle, enable a CUDA GPU and Internet, attach the licensed datasets, then run the cells in order. The notebook clones the selected GitHub revision, creates the Kaggle Python environment at a fixed path, builds the Java engine, validates CUDA/DALI, and keeps pilot, confirmatory, DALI, and systems measurements separate. It does not create paper results without executing those campaigns.

In [ ]:
from pathlib import Path
import json
import os
import subprocess

REPOSITORY = 'https://github.com/Grgur00/Aether-Engine.git'
GIT_REF = 'main'  # Replace with a frozen commit SHA before confirmatory measurements.
REPO = Path('/kaggle/working/Aether-Engine')
RESULTS = Path('/kaggle/working/aether-results')
PY = '/kaggle/working/aether-paper-venv/bin/python'

# Replace these with manifests prepared from your attached, licensed datasets.
OCT_V1 = '/kaggle/working/aether-data/oct5k-v1.csv'
OCT_V2 = '/kaggle/working/aether-data/oct5k-v2.csv'
COCO_V1 = '/kaggle/working/aether-data/coco-v1.csv'
COCO_V2 = '/kaggle/working/aether-data/coco-v2.csv'
IMAGENET_V1 = '/kaggle/working/aether-data/imagenet-v1.csv'
IMAGENET_V2 = '/kaggle/working/aether-data/imagenet-v2.csv'

RUN_CONFIRMATORY = False
RUN_DALI_SMOKE = False
RUN_DALI_COMPARISON = False
RUN_SYSTEMS_CAMPAIGNS = False

In [ ]:
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', '--tags', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', GIT_REF], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', GIT_REF, REPOSITORY, str(REPO)], check=True)

subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], check=True)
subprocess.run(['bash', str(REPO / 'kaggle/setup.sh')], cwd=REPO, check=True)
runtime = json.loads(Path('/kaggle/working/aether-paper-runtime.json').read_text())
os.environ['JAVA_HOME'] = runtime['javaHome']
os.environ['PATH'] = runtime['javaHome'] + '/bin:' + os.environ['PATH']
os.environ['PYTHONPATH'] = f"{REPO / 'clients/python'}:{REPO / 'scripts'}"
subprocess.run([PY, str(REPO / 'scripts/validate_gpu.py')], cwd=REPO, check=True)


def run(script, *arguments):
    subprocess.run([PY, str(REPO / 'scripts' / script), *map(str, arguments)], cwd=REPO, check=True)

## Smoke checks
This runs real Java storage, worker processes, actual CPU optimization, source/transform invalidation and externally killed writers. Tiny generated fixtures are correctness tests, not dataset performance evidence. Use a new smoke output directory when rerunning the whole suite.


In [ ]:
run('reproduce.py', 'smoke', '--output', RESULTS / 'smoke')


## Real data configuration
Existing OCT manifests should preserve the original V1/V2 membership for direct replication. If only a full manifest exists, use `scripts/prepare_evolution.py` as described in `kaggle/README.md`. The runner verifies manifest/source checksums and fails if paths or counts are wrong.


In [ ]:
for path in (OCT_V1, OCT_V2):
    if not Path(path).is_file():
        raise FileNotFoundError(f'Update OCT_V1/OCT_V2 or prepare this manifest: {path}')

config = json.loads((REPO / 'configs/paper/datasets.json').read_text())
config['oct5k'].update(manifestV1=OCT_V1, manifestV2=OCT_V2)
vision_manifests = {
    'coco': (COCO_V1, COCO_V2),
    'imagenet': (IMAGENET_V1, IMAGENET_V2),
}
for dataset, (v1, v2) in vision_manifests.items():
    present = [Path(path).is_file() for path in (v1, v2)]
    if any(present) and not all(present):
        raise FileNotFoundError(f'{dataset} requires both V1 and V2 manifests')
    if all(present):
        config[dataset].update(manifestV1=v1, manifestV2=v2)

CONFIG = Path('/kaggle/working/aether-datasets.json')
CONFIG.write_text(json.dumps(config, indent=2))
run('validate_manifests.py', '--config', CONFIG, '--datasets', 'oct5k')
print('Validated OCT5K manifests. COCO/ImageNet are validated immediately before DALI execution.')

## Separate 10-block pilot
Each block uses fresh stores, restarts Java after V1 population, and measures V2 in randomized backend order. `--resume` verifies raw evidence and the frozen protocol; different hardware/software needs a new campaign. Do not run two drivers for the same output at once.


In [ ]:
run('run_matrix.py', '--config', CONFIG, '--repeats', 10, '--resume', '--output', RESULTS / 'pilot')
run('analyze.py', '--input', RESULTS / 'pilot', '--pilot', '--output', RESULTS / 'pilot-analysis')
run('figures.py', '--input', RESULTS / 'pilot', '--output', RESULTS / 'pilot-figures')


## Confirmatory campaign after code/protocol freeze
Keep this disabled until source review/commit and pilot sample-size planning are finished. A dirty archive is intentionally rejected by `--confirmatory`. The source and timing protocol must remain fixed across all primary blocks.


In [ ]:
PRIMARY_REPEATS = 24  # Revise from the separate pilot before starting this campaign.
if RUN_CONFIRMATORY:
    if not provenance['sourceClean']:
        raise RuntimeError('Confirmatory measurement requires a clean, committed source archive')
    run('run_matrix.py', '--config', CONFIG, '--datasets', 'oct5k', '--repeats', PRIMARY_REPEATS,
        '--confirmatory', '--resume', '--output', RESULTS / 'primary')
    run('analyze.py', '--input', RESULTS / 'primary', '--holm', '--output', RESULTS / 'primary/processed')
    run('figures.py', '--input', RESULTS / 'primary', '--output', RESULTS / 'primary/figures')

## DALI CUDA validation and secondary comparison

Install DALI only for this distinct CUDA workload. Its generated-fixture smoke establishes integration correctness but is rejected as research evidence. The 12-block comparison uses real COCO and ImageNet V1/V2 manifests, validates them before measurement, and writes results to a separate directory. These results must never be pooled with the Pillow/OCT primary experiment.

In [ ]:
if RUN_DALI_SMOKE or RUN_DALI_COMPARISON:
    subprocess.run([PY, '-m', 'pip', 'install', '-r', str(REPO / 'env/requirements-dali.lock')], check=True)
    run('validate_gpu.py', '--require-dali')

if RUN_DALI_SMOKE:
    run('dali_smoke.py', '--output', RESULTS / 'dali-smoke')

if RUN_DALI_COMPARISON:
    run('validate_manifests.py', '--config', CONFIG, '--datasets', 'coco,imagenet')
    run('dali_comparison.py', '--config', CONFIG, '--datasets', 'coco,imagenet', '--repeats', 12,
        '--resume', '--output', RESULTS / 'dali')
    run('dali_analyze.py', '--input', RESULTS / 'dali', '--output', RESULTS / 'dali/processed')

## Optional systems campaigns and evidence preservation

Run systems campaigns only after the pilot and with sufficient remote disk/time. Worker scaling, client concurrency, failure injection, and transform evolution are independent evidence families. Every result directory, manifest, protocol, environment receipt, and analysis output must be retained together.

In [ ]:
if RUN_SYSTEMS_CAMPAIGNS:
    run('run_matrix.py', '--config', CONFIG, '--datasets', 'oct5k', '--workers', '0,2,4,8', '--repeats', 10,
        '--resume', '--output', RESULTS / 'worker-scaling')
    run('concurrency_matrix.py', '--clients', '1,2,4', '--workers', '0,2,4,8', '--repeats', 10,
        '--output', RESULTS / 'concurrency')
    run('fault_injection.py', '--trials-per-point', 100, '--output', RESULTS / 'durability')
    run('transform_evolution.py', '--repeats', 10, '--output', RESULTS / 'transform-evolution')

if RESULTS.exists():
    run('checksums.py', RESULTS)
    run('checksums.py', RESULTS, '--verify')
    print('Save this directory as Kaggle notebook output:', RESULTS)
else:
    print('No results directory exists yet.')